In [1]:
import requests
import pandas as pd
import time
import yfinance as yf
from bs4 import BeautifulSoup
import numpy as np

pd.set_option('display.max_colwidth', None)
pd.options.display.max_rows = 200

In [2]:
session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/121.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://br.advfn.com/",
    "Connection": "keep-alive",
})

In [3]:
precos_teto = {
    "WIZC3": 10.00, "BBSE3": 35.50, "BBAS3": 25.00, "UNIP6": 70.00, "SEER3": 14.00, "VALE3": 75.00,
    "PETR4": 34.00, "AXIA6": 43.80, "TUPY3": 21.00, "AGRO3": 27.50, "EGIE3": 28.60, "ITSA4": 9.50,
    "VAMO3": 10.90, "B3SA3": 17.00, "KLBN4": 5.60, "TTEN3": 14.70, "PRIO3": 62.75, "BRBI11": 18.00,
    "PNVL3": 12.00, "SIMH3": 10.00, "GMAT3": 7.12, "TIMS3": 18.60, "VIVA3": 25.00, "EZTC3": 13.29
}

OPTIONS_URL = "https://br.advfn.com/common/bov-options/api"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/121.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://br.advfn.com/",
}

EXPIRY = "2026-05"


# -------------------------------
# Funções auxiliares
# -------------------------------
def to_float(valor):
    if valor is None:
        return None
    if isinstance(valor, (int, float)):
        return float(valor)
    return float(valor.replace(".", "").replace(",", "."))


def get_preco_atual(ativo):
    ticker = yf.Ticker(f"{ativo}.SA")
    hist = ticker.history(period="1d")

    if hist.empty:
        return None

    return float(hist["Close"].iloc[-1])


# -------------------------------
# Coleta de dados
# -------------------------------
linhas = []

for ativo, preco_teto in precos_teto.items():
    # print(f"Buscando opções de {ativo}...")

    try:
        preco_atual = get_preco_atual(ativo)

        params = {
            "symbol": ativo,
            "expiry_date": EXPIRY
        }

        r = requests.get(OPTIONS_URL, params=params, headers=HEADERS)

        if r.status_code != 200:
            print(f"Erro ao buscar {ativo}")
            continue

        data = r.json().get("result", [])

        for opcao in data:
            linhas.append({
                # Identificação
                "ativo": ativo,
                "opcao": opcao.get("symbol"),
                "tipo": opcao.get("type"),        # Put / Call
                "estilo": opcao.get("style"),     # E = europeia

                # Preços
                "strike": to_float(opcao.get("strike_price")),
                "preco_opcao": to_float(opcao.get("last_price")),
                "preco_atual": preco_atual,
                "preco_teto": preco_teto,

                # Datas
                "vencimento": opcao.get("expiry_date"),

                # Liquidez e variação
                "volume": opcao.get("volume"),
                "volume_formatado": opcao.get("volume_form"),
                "variacao_percentual": opcao.get("change_percentage"),

                # Link direto
                "url": f"https://br.advfn.com" + opcao.get('url')

            })

        time.sleep(0.3)

    except Exception as e:
        print(f"Erro em {ativo}: {e}")


df_base = pd.DataFrame(linhas)

In [4]:
df_base

,ativo,opcao,tipo,estilo,strike,preco_opcao,preco_atual,preco_teto,vencimento,volume,volume_formatado,variacao_percentual,url
0,WIZC3,WIZCE60,Call,A,5.43,None,9.440000,10.00,15/05/2026,0,0,"0,00%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EWIZCE60
1,WIZC3,WIZCQ60,Put,E,5.43,None,9.440000,10.00,15/05/2026,0,0,"0,00%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EWIZCQ60
2,BBSE3,BBSEE364,Call,E,36.43,None,34.330002,35.50,15/05/2026,1100,1.100,"9,68%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EBBSEE364
3,BBSE3,BBSEQ335,Put,E,30.93,None,34.330002,35.50,15/05/2026,300,300,"-27,78%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EBBSEQ335
4,BBSE3,BBSEQ176,Put,E,17.68,None,34.330002,35.50,15/05/2026,0,0,"0,00%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EBBSEQ176
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2149,EZTC3,EZTCE150,Call,A,15.03,None,14.390000,13.29,15/05/2026,0,0,"0,00%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EEZTCE150
2150,EZTC3,EZTCE160,Call,A,16.03,None,14.390000,13.29,15/05/2026,0,0,"0,00%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EEZTCE160
2151,EZTC3,EZTCQ170,Put,E,17.03,None,14.390000,13.29,15/05/2026,0,0,"0,00%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EEZTCQ170
2152,EZTC3,EZTCE170,Call,A,17.03,None,14.390000,13.29,15/05/2026,0,0,"0,00%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EEZTCE170


In [5]:
df = df_base[
    ((df_base['strike']/df_base['preco_atual']).between(0.85,1)) &
    (df_base['tipo'] == 'Put') &
    (df_base['volume'] > 0 )
]

df

,ativo,opcao,tipo,estilo,strike,preco_opcao,preco_atual,preco_teto,vencimento,volume,volume_formatado,variacao_percentual,url
3,BBSE3,BBSEQ335,Put,E,30.93,None,34.330002,35.50,15/05/2026,300,300,"-27,78%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EBBSEQ335
138,BBAS3,BBASQ229,Put,E,22.75,None,24.830000,25.00,15/05/2026,10000,10.000,"-23,64%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EBBASQ229
139,BBAS3,BBASQ239,Put,E,23.75,None,24.830000,25.00,15/05/2026,5000,5.000,"-31,25%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EBBASQ239
142,BBAS3,BBASQ215,Put,E,21.25,None,24.830000,25.00,15/05/2026,1400,1.400,"-4,76%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EBBASQ215
150,BBAS3,BBASQ228,Put,E,22.50,None,24.830000,25.00,15/05/2026,200,200,"-7,50%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EBBASQ228
450,VALE3,VALEQ754,Put,E,75.40,None,80.050003,75.00,15/05/2026,4000,4.000,"-31,58%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EVALEQ754
452,VALE3,VALEQ779,Put,E,77.90,None,80.050003,75.00,15/05/2026,900,900,"-19,22%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EVALEQ779
453,VALE3,VALEQ737,Put,E,70.40,None,80.050003,75.00,15/05/2026,900,900,"-40,13%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EVALEQ737
458,VALE3,VALEQ727,Put,E,69.40,None,80.050003,75.00,15/05/2026,200,200,"-37,01%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EVALEQ727
460,VALE3,VALEQ744,Put,E,74.40,None,80.050003,75.00,15/05/2026,200,200,"0,00%",https://br.advfn.com/p.php?pid=quote&symbol=BOV%5EVALEQ744


In [6]:
toFloat = ['strike', 'preco_opcao','preco_atual', 'preco_teto']

for i in toFloat:
    df[i] = df[i].astype(float)

toString = ['ativo', 'opcao','tipo', 'estilo','url']

for i in toString:
    df[i] = df[i].astype(str)

C:\Users\FamiliaUchoa\AppData\Local\Temp\ipykernel_13520\1949950904.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[i] = df[i].astype(float)
C:\Users\FamiliaUchoa\AppData\Local\Temp\ipykernel_13520\1949950904.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[i] = df[i].astype(float)
C:\Users\FamiliaUchoa\AppData\Local\Temp\ipykernel_13520\1949950904.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value in

In [7]:
def buscar_preco_opcao(url):
    try:
        r = session.get(url, timeout=15)
        r.raise_for_status()

        soup = BeautifulSoup(r.text, "html.parser")

        span = soup.select_one("span.cur-price")
        if not span:
            print("⚠️ span.cur-price não encontrado")
            return None

        preco_txt = span.text.strip()

        return float(preco_txt.replace(".", "").replace(",", "."))

    except Exception as e:
        print(f"Erro ao buscar preço da opção: {url} | {e}")
        return None
    
for i, row in df.iterrows():
    print(f"Buscando preço da opção {row['opcao']}...")

    preco = buscar_preco_opcao(row["url"])
    df.at[i, "preco_opcao"] = preco

    time.sleep(0.3)  # MUITO importante para não tomar block

Buscando preço da opção BBSEQ335...
Buscando preço da opção BBASQ229...
Buscando preço da opção BBASQ239...
Buscando preço da opção BBASQ215...
Buscando preço da opção BBASQ228...
Buscando preço da opção VALEQ754...
Buscando preço da opção VALEQ779...
Buscando preço da opção VALEQ737...
Buscando preço da opção VALEQ727...
Buscando preço da opção VALEQ744...
Buscando preço da opção VALEQ764...
Buscando preço da opção PETRQ420...
Buscando preço da opção PETRQ380...
Buscando preço da opção PETRQ367...
Buscando preço da opção PETRQ392...
Buscando preço da opção PETRQ390...
Buscando preço da opção PETRQ37...
Buscando preço da opção PETRQ410...
Buscando preço da opção ITSAQ134...
Buscando preço da opção ITSAQ129...
Buscando preço da opção B3SAQ155...
Buscando preço da opção B3SAQ165...
Buscando preço da opção B3SAQ167...
Buscando preço da opção B3SAQ172...
Buscando preço da opção B3SAQ163...
Buscando preço da opção B3SAQ175...
Buscando preço da opção B3SAQ160...
Buscando preço da opção PRIOQ

In [14]:
# parametrizado para CALL e PUT

aporte = 5500

# rentabilidade
df['rentabilidade'] = round((df['preco_opcao'] / df['strike']) * 100, 2)

# distancia
df['distancia'] = round(df['strike'] / df['preco_atual'], 3)

# strike com premio
df['strike - premio'] = df['strike'] - df['preco_opcao']
df['strike + premio'] = df['strike'] + df['preco_opcao']

# intrinseco e extrinseco
df['intrinsecoPUT'] = (df['strike'] - df['preco_atual']).clip(lower=0)
df['extrinsecoPUT'] = abs(df['preco_opcao'] - df['intrinsecoPUT'])

df['intrinsecoCALL'] = (df['preco_atual'] - df['strike'] ).clip(lower=0)
df['extrinsecoCALL'] = abs(df['preco_opcao'] - df['intrinsecoCALL'])


df['cotas'] = np.floor((aporte / df['strike'])/100) * 100
df['premio X cotas'] = df['cotas'] * df['preco_opcao']


df['preco_atual'] = round(df['preco_atual'],2)

C:\Users\FamiliaUchoa\AppData\Local\Temp\ipykernel_13520\903914789.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['rentabilidade'] = round((df['preco_opcao'] / df['strike']) * 100, 2)
C:\Users\FamiliaUchoa\AppData\Local\Temp\ipykernel_13520\903914789.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['distancia'] = round(df['strike'] / df['preco_atual'], 3)
C:\Users\FamiliaUchoa\AppData\Local\Temp\ipykernel_13520\903914789.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of 

In [15]:
df_put = df[

    (df['tipo'] == 'Put') &
    (df['distancia'] <= 0.95) &
    (df['preco_teto'] >= df['preco_atual'])

][['opcao', 'tipo', 'preco_atual', 'strike', 'distancia', 'preco_opcao',
    'strike - premio', 'preco_teto', 'variacao_percentual', 'rentabilidade',
    'cotas', 'premio X cotas', 'vencimento']].sort_values('rentabilidade', ascending=False)

df_put

,opcao,tipo,preco_atual,strike,distancia,preco_opcao,strike - premio,preco_teto,variacao_percentual,rentabilidade,cotas,premio X cotas,vencimento
1724,PRIOQ545,Put,58.89,54.50,0.925,1.88,52.62,62.75,"0,00%",3.45,100.0,188.0,15/05/2026
1733,PRIOQ54,Put,58.89,54.00,0.917,1.70,52.30,62.75,"41,67%",3.15,100.0,170.0,15/05/2026
1723,PRIOQ535,Put,58.89,53.50,0.908,1.62,51.88,62.75,"58,82%",3.03,100.0,162.0,15/05/2026
1716,PRIOQ52,Put,58.89,52.00,0.883,1.26,50.74,62.75,"53,66%",2.42,100.0,126.0,15/05/2026
1732,PRIOQ525,Put,58.89,52.50,0.891,1.24,51.26,62.75,"39,33%",2.36,100.0,124.0,15/05/2026
1729,PRIOQ505,Put,58.89,50.50,0.858,0.99,49.51,62.75,"52,31%",1.96,100.0,99.0,15/05/2026
138,BBASQ229,Put,24.83,22.75,0.916,0.42,22.33,25.00,"-23,64%",1.85,200.0,84.0,15/05/2026
150,BBASQ228,Put,24.83,22.50,0.906,0.37,22.13,25.00,"-7,50%",1.64,200.0,74.0,15/05/2026
142,BBASQ215,Put,24.83,21.25,0.856,0.20,21.05,25.00,"-4,76%",0.94,200.0,40.0,15/05/2026
3,BBSEQ335,Put,34.33,30.93,0.901,0.13,30.80,35.50,"-27,78%",0.42,100.0,13.0,15/05/2026


In [10]:
df_call = df[

    (df['tipo'] == 'Call') &
    (df['distancia'] >= 1.04)

][['opcao', 'tipo', 'preco_atual', 'strike', 'distancia', 'preco_opcao',
    'strike + premio', 'preco_teto', 'variacao_percentual', 'rentabilidade',
    'cotas', 'premio X cotas', 'vencimento']].sort_values('rentabilidade', ascending=False)

df_call

,opcao,tipo,preco_atual,strike,distancia,preco_opcao,strike + premio,preco_teto,variacao_percentual,rentabilidade,cotas,premio X cotas,vencimento


In [11]:
df_put.describe().to_html('vendaDePutDescribe.html')
df_call.describe().to_html('vendaDeCallDescribe.html')

df_put.to_html('vendaDePut.html')
df_call.to_html('vendaDeCall.html')

In [12]:
df

,ativo,opcao,tipo,estilo,strike,preco_opcao,preco_atual,preco_teto,vencimento,volume,...,rentabilidade,distancia,strike - premio,strike + premio,intrinsecoPUT,extrinsecoPUT,intrinsecoCALL,extrinsecoCALL,cotas,premio X cotas
3,BBSE3,BBSEQ335,Put,E,30.93,0.13,34.33,35.50,15/05/2026,300,...,0.42,0.901,30.80,31.06,0.0,0.13,3.400002e+00,3.270002,300.0,39.0
138,BBAS3,BBASQ229,Put,E,22.75,0.42,24.83,25.00,15/05/2026,10000,...,1.85,0.916,22.33,23.17,0.0,0.42,2.080000e+00,1.660000,500.0,210.0
139,BBAS3,BBASQ239,Put,E,23.75,0.55,24.83,25.00,15/05/2026,5000,...,2.32,0.957,23.20,24.30,0.0,0.55,1.080000e+00,0.530000,500.0,275.0
142,BBAS3,BBASQ215,Put,E,21.25,0.20,24.83,25.00,15/05/2026,1400,...,0.94,0.856,21.05,21.45,0.0,0.20,3.580000e+00,3.380000,500.0,100.0
150,BBAS3,BBASQ228,Put,E,22.50,0.37,24.83,25.00,15/05/2026,200,...,1.64,0.906,22.13,22.87,0.0,0.37,2.330000e+00,1.960000,500.0,185.0
450,VALE3,VALEQ754,Put,E,75.40,1.88,80.05,75.00,15/05/2026,4000,...,2.49,0.942,73.52,77.28,0.0,1.88,4.650003e+00,2.770003,100.0,188.0
452,VALE3,VALEQ779,Put,E,77.90,2.27,80.05,75.00,15/05/2026,900,...,2.91,0.973,75.63,80.17,0.0,2.27,2.150003e+00,0.119997,100.0,227.0
453,VALE3,VALEQ737,Put,E,70.40,0.94,80.05,75.00,15/05/2026,900,...,1.34,0.879,69.46,71.34,0.0,0.94,9.650003e+00,8.710003,100.0,94.0
458,VALE3,VALEQ727,Put,E,69.40,0.80,80.05,75.00,15/05/2026,200,...,1.15,0.867,68.60,70.20,0.0,0.80,1.065000e+01,9.850003,100.0,80.0
460,VALE3,VALEQ744,Put,E,74.40,1.31,80.05,75.00,15/05/2026,200,...,1.76,0.929,73.09,75.71,0.0,1.31,5.650003e+00,4.340003,100.0,131.0
